# BangHallu — Neural models on GPU (M3 + M13)

Trains the five from-scratch neural models on a Colab GPU instead of a laptop CPU.

| Model | What it is | Lab |
|---|---|---|
| `rnn` | vanilla RNN, forward only | 4 |
| `birnn` | bidirectional RNN | 4 |
| `bilstm` | 2-layer bidirectional LSTM | 4 |
| `bilstm_attn` | BiLSTM + dot-product attention | 4 |
| `transformer_scratch` | Transformer encoder, **no pretraining** | 5 |

**Nothing about the models changes here — only where the arithmetic runs.** The training code
lives in `src/train_neural.py` and is imported, not copied. A notebook that re-implemented it
would drift from the tested version the first time either was edited.

**Before you run:** *Runtime → Change runtime type → T4 GPU*.

**The test split is never touched.** These models train on `train` and are scored on `dev`.
`test` is opened exactly once, at the very end of the project.

## 1. Mount Drive and find the project

In [ ]:
# Put the project folder in your Drive first (it needs src/ and data/splits/),
# for example at MyDrive/Bhibranti. Then run this cell.
import sys, pathlib

REPO = "/content/drive/MyDrive/Bhibranti"      # <- change if you put it somewhere else

sys.path.insert(0, str(pathlib.Path(REPO) / "src"))
from colab_setup import prepare

paths = prepare(repo=REPO)

The `data hash` printed above should match what your laptop prints for the same command.
If it does not, Drive is holding an older copy of the splits and every number below would be
incomparable with the ones already in `results/experiment_log.csv`.

## 2. Check the GPU is really being used

In [ ]:
import torch
import train_neural as tn

device = tn.pick_device()
print("device:", tn.describe_device(device))
assert device.type == "cuda", "No GPU. Runtime > Change runtime type > T4 GPU, then re-run."

## 3. Prove the three fixes to the lab code still hold

`--check` is not a formality. It demonstrates, on real dev records, three mistakes in the lab
notebooks that would each quietly produce a worse model rather than an error message:

1. **Padding corrupts the RNN's final state.** Lab 4 reads the state at the last position of a
   padded batch, so a short record's "summary of the sentence" is really the state after the
   network has finished running over `<PAD>`.
2. **Lab 5 averages the Transformer's output over the padding too** — harmless for its 8-word
   toy sentences, wrong for our 3–250 token inputs.
3. **Lab 5 scales embeddings by `sqrt(d_model)` but leaves PyTorch's default `N(0,1)` start**,
   which makes token vectors about 11× larger than the position signal, so word order is
   effectively drowned out.

Every line it prints is measured, not asserted.

In [ ]:
import subprocess, sys

done = subprocess.run([sys.executable, "src/train_neural.py", "--check"],
                      cwd=paths.root, capture_output=True, text=True)
print(done.stdout)
assert done.returncode == 0, "a rule failed — do not trust any score below until it passes"

## 4. One record through every model, before committing to a long run

In [ ]:
done = subprocess.run([sys.executable, "src/train_neural.py", "--demo"],
                      cwd=paths.root, capture_output=True, text=True)
print(done.stdout)

## 5. Train one model first

Always do this before the full sweep: it takes a couple of minutes and catches a broken setup
early.

Watch for **dev loss stopping its improvement after a few epochs while train loss keeps
falling**. That is the model memorising 6,134 records, and it is exactly why training stops
itself and goes back to the best epoch rather than keeping the last one.

In [ ]:
from splits import load_split

train, dev = load_split("train"), load_split("dev")
vocab = tn.build_vocabulary(train)
vectors = tn.load_or_train_vectors(42, "V1")    # the M2 Skip-gram; trains once if missing

result, target, notes, fitted = tn.run_one(
    "bilstm_attn", 42, "F2", "V1", train, dev, vocab, vectors, device=device)

## 6. The full sweep: 5 models × 3 seeds

On a T4 this is roughly 10–20 minutes. The same sweep takes hours on a laptop CPU.

`write_log=True` appends every run to `results/experiment_log.csv`. Because the repo lives on
Drive, that file survives the session ending — which is the reason for mounting Drive rather
than uploading loose files.

In [ ]:
tn.run_all(seeds=(42, 1337, 2024), fmt="F2", variant="V1",
           write_log=True, note="Colab GPU run", device=device)

## 7. Where the attention model looked

This is the only model on the ladder that can show *where* it looked. It reports how attention
split across passage / question / answer for 20 dev records — 10 correct, 10 hallucinated,
half of each hard.

Read the result carefully: attention describes what the model **used**, not that it reasoned.

In [ ]:
import evaluate

rows = tn.attention_report(fitted, dev)     # `fitted` is the bilstm_attn from section 5
tn.print_attention(rows)
print(evaluate.write_table(rows, "attention_bilstm_attn_s42_gpu.csv"))

## 8. Did anything actually beat the no-learning rules?

A model that cannot beat a two-line string-matching rule has demonstrated nothing. On the hard
subset the bar is **0.591** (the fuzzy rule), not 0.487 (the exact one).

McNemar is used instead of eyeballing the gap: two scores differing by 0.01 on 218 records is
usually noise, and the guide requires the test before any claim that one model beats another.

In [ ]:
hard = [i for i, r in enumerate(dev) if r["context"] and r["difficulty"] == "hard"]
truth = [dev[i]["label"] for i in hard]

all_predictions = fitted.predict(dev)
model_pred = [all_predictions[i] for i in hard]
fuzzy_all = evaluate.baseline_predictions(dev, "fuzzy")
fuzzy = [fuzzy_all[i] for i in hard]

wins_model, wins_rule, p = evaluate.mcnemar(truth, model_pred, fuzzy)
print(f"on {len(hard)} hard, has-context dev records:")
print(f"  this model  {evaluate.macro_f1(truth, model_pred):.3f}")
print(f"  fuzzy rule  {evaluate.macro_f1(truth, fuzzy):.3f}")
print(f"  they disagree on {wins_model + wins_rule}: model right {wins_model}, rule right {wins_rule}")
print(f"  McNemar p = {p:.3f}  ->", "too close to call" if p > 0.05 else "a real difference")

low, high = evaluate.bootstrap_interval([dev[i] for i in hard], model_pred)
print(f"  95% interval (PAIRS resampled, not records): {low:.3f} to {high:.3f}")

## 9. What to expect, and what not to

These models have **no pretraining**. On ~6,000 records they land in the 0.50–0.65 band, which
guide §12.3 predicts and which is the correct outcome, not a failure to be tuned away.

The Transformer in particular should land near the classical models: the same architecture
family as BanglaBERT with none of its pretraining. **That gap is the finding.**

One honest caveat on every number here: dev decides when training stops and which epoch is
kept, so these dev scores are mildly optimistic. `test` remains untouched.

Next: **`02_bert_gpu.ipynb`**, which fine-tunes the pretrained encoders (M4/M5) and is where
the scores should finally move.